# Day 7: RAG Earnings Chatbot

**Goal:** turn the Day-5 earnings-call embeddings into a question-answering chatbot, then put it online.

By the end you can: chunk a set of earnings transcripts into passages, embed them, retrieve the top-k passages closest to a question, build a prompt that contains only those passages, and let a small local language model write a grounded answer that cites which passage it came from. Then you deploy that pipeline to a free Hugging Face Space.

**Connection to Day 5:** the earnings snippets and the embedding model (`paraphrase-MiniLM-L3-v2`, local, no API key) are exactly the ones from `Day5_Day6_Embeddings.ipynb`. Day 5 measured which companies talk about similar things. Day 7 uses that same similarity to fetch the right text before a model answers a question about it.

<!-- # [[FULL]] -->

## Glossary (Day 7 RAG terms)

Read this once. The notebook will refer back to each term as it comes up.

| Term | Plain English | Finance analogy |
|---|---|---|
| **RAG** | Retrieval-Augmented Generation: fetch the relevant text, then let a model answer using it. | An analyst who pulls the transcript passage before answering, not from memory. |
| **Retrieval** | Find the most relevant chunks of text for a given question. | The research assistant who locates the right page in the 10-K. |
| **Generation** | A language model writes the answer in natural language. | The analyst who phrases the answer in plain English after reading the source. |
| **Embedding** | A list of numbers (a vector) that represents text meaning. | A risk score for text: similar texts get similar vectors. |
| **Chunk** | A short passage of text (one sentence to a paragraph) treated as a retrieval unit. | A single paragraph from an earnings call transcript. |
| **Top-k** | Keep only the k closest chunks (e.g., k=3) instead of all. | Show the analyst the 3 most relevant passages, not the whole transcript. |
| **Cosine similarity** | A number 0 to 1 measuring how similarly two vectors point. | How alike two filings are in topic, regardless of length. |
| **Prompt** | The text we send to the language model (context + question). | The briefing memo handed to the analyst before they answer. |
| **Context window** | The maximum text the language model can read at once. | The desk space limit: only so many pages fit at one time. |
| **Hallucination** | When the model invents a fact that is not in the source. | An analyst guessing a revenue number they did not look up. |
| **Citation** | Pointing the answer to the exact chunk it came from. | The footnote linking the claim to a specific transcript line. |
| **flan-t5-small** | A small open-source language model (~300 MB) that runs on Colab CPU. | A junior analyst: not the most eloquent, but free, fast, and traceable. |


## Setup: install the libraries

The next cell installs `sentence-transformers` (the embedding model from Day 5), `transformers` and `torch` (to run the small text-generation model), and `gradio` (the web interface we deploy at the end). Run it once; the first run takes about a minute. `# [FULL]`

In [1]:
!pip -q install sentence-transformers transformers torch gradio
print('RAG stack installed.')

RAG stack installed.


## Step 1: Rebuild the Day-5 setup (verbatim)

The cells in this step are copied from `Day5_Day6_Embeddings.ipynb` (commit `0a7ad3a9`): the imports and the embedding model, then the six synthetic earnings-call snippets. We re-run them here so the notebook is self-contained. The first cell downloads the small `paraphrase-MiniLM-L3-v2` model (about 20 seconds on first run). The pip install (the first line of the Day-5 setup cell) is hoisted into its own cell just above so the rest of the setup cell is plain Python.

In [2]:
!pip install -q sentence-transformers

In [3]:
# source: Day5_Day6_Embeddings.ipynb @ 0a7ad3a9
# Cell 0: Setup -- run this first (~20 s on first run)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

np.random.seed(42)

model = SentenceTransformer('paraphrase-MiniLM-L3-v2')
print(f'Ready. Embedding dimension: {model.get_sentence_embedding_dimension()}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ready. Embedding dimension: 384


/tmp/ipykernel_12440/1817652646.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Ready. Embedding dimension: {model.get_sentence_embedding_dimension()}')


In [4]:
# source: Day5_Day6_Embeddings.ipynb @ 0a7ad3a9
# Six synthetic earnings call snippets -- read these before starting the exercises
snippets = [
    # 0: TechCore (technology)
    ('TechCore delivered record cloud revenue this quarter, growing 34 percent year-over-year. '
     'Our AI inference platform saw adoption across 12 new enterprise clients. '
     'Capital expenditure for data center expansion reached 2.1 billion dollars. '
     'We guide for continued double-digit growth as enterprises accelerate digital transformation. '
     'Operating margins improved 200 basis points driven by software mix shift.'),

    # 1: ChipMaker (technology)
    ('ChipMaker reported strong demand for high-bandwidth memory in AI accelerator markets. '
     'Our next-generation GPU architecture enters mass production next quarter. '
     'Revenue from the data center segment grew 58 percent versus prior year. '
     'We are expanding fab capacity with a 4 billion dollar investment. '
     'Supply constraints in advanced packaging remain our primary near-term risk.'),

    # 2: FirstBank (banking)
    ('FirstBank reported net interest income of 3.4 billion, up 8 percent on higher deposit spreads. '
     'Credit quality remains solid with non-performing loans at 0.7 percent of total portfolio. '
     'We grew our commercial lending book by 12 percent while maintaining disciplined underwriting. '
     'Tier 1 capital ratio stands at 13.2 percent, well above regulatory minimums. '
     'Provision for credit losses declined as charge-offs trended favorably.'),

    # 3: RegionalBank (banking)
    ('RegionalBank navigated a challenging rate environment with net interest margin compression. '
     'We added 42,000 new checking accounts, demonstrating strong retail franchise strength. '
     'Mortgage origination volume fell 23 percent as refinancing activity slowed. '
     'Cost efficiency ratio improved to 58 percent following branch optimization. '
     'We remain well-capitalized with CET1 ratio of 12.8 percent.'),

    # 4: OilGiant (energy)
    ('OilGiant achieved production of 1.2 million barrels per day, exceeding guidance by 4 percent. '
     'Realized oil prices averaged 82 dollars per barrel, supporting strong free cash flow. '
     'We returned 2.8 billion dollars to shareholders through buybacks and dividends. '
     'Exploration success in the Gulf adds 200 million barrels to proven reserves. '
     'Refining margins benefited from tight distillate markets globally.'),

    # 5: SolarPower (energy)
    ('SolarPower commissioned 3.2 gigawatts of new solar capacity across seven markets. '
     'Revenue grew 41 percent driven by long-term power purchase agreements. '
     'Our battery storage pipeline reached 18 gigawatt-hours, enabling round-the-clock delivery. '
     'Government incentives contributed 340 million dollars to project economics. '
     'We are on track to reach 20 gigawatts of total installed capacity by year-end.'),
]

companies = ['TechCore', 'ChipMaker', 'FirstBank', 'RegionalBank', 'OilGiant', 'SolarPower']
print('Loaded', len(snippets), 'earnings call snippets for:', companies)


Loaded 6 earnings call snippets for: ['TechCore', 'ChipMaker', 'FirstBank', 'RegionalBank', 'OilGiant', 'SolarPower']


<!-- # [[FULL]] -->

### Section 1 terms (just-in-time recap)

- **Chunk**: one short passage. We will split each earnings call into a few chunks.
- **Embedding**: vector of 384 numbers for each chunk.
- **Cosine similarity**: tells us which chunks are close in meaning to a query.
- **Top-k**: keep only the k=3 highest-similarity chunks; drop the rest.

## Step 2: What is RAG? (read this before any code)

**RAG = Retrieval-Augmented Generation. Retrieve the relevant earnings snippets, stuff them into the prompt, let the model answer grounded in real text, not hallucinated. Retrieval is the half you can trust; generation is the half that needs grounding.**

Why bother? Ask a plain language model "what was TechCore's cloud revenue growth?" and it will produce a fluent, confident, possibly invented number, because it never read TechCore's transcript. It does not say "I do not know"; it generates the most plausible-sounding continuation. In finance that is disqualifying: an analyst cannot act on a number a model made up.

The fix is not a bigger model. The fix is to hand the model the relevant text first. RAG has two halves:

- **Retrieval** (the trusted half): embed the snippets, embed the question, find the snippets closest in meaning. This is exactly the cosine-similarity skill from Day 5. No model can hallucinate here; it is just distance arithmetic over vectors.
- **Generation** (the half that needs grounding): a small language model writes the final answer, but only from the retrieved snippets placed in its prompt, not from its training memory. The model can phrase things naturally, but it is anchored to text you can show the analyst.

> **Finance analogy:** a bare language model is an analyst answering from memory under time pressure: fluent, but unverifiable. A RAG system is the same analyst who, before answering, pulls the exact transcript passages and reads from them.

### Instructor demo: retrieval over three toy sentences

Before the full pipeline, here is retrieval at its smallest: three short sentences, one query, find the closest. Embed all four, take cosine similarities, print the winner. This is the whole "retrieval" half of RAG in eight lines. The full pipeline below just does this over real earnings text instead of toy sentences, and then feeds the winner to a generator.

<!-- # [[FULL]] -->

**Expected output (the toy retrieval demo below):**

- The closest match to the query `"company revenue grew"` is the sentence about revenue growth.
- The cosine similarity for that match is roughly between 0.4 and 0.8 (depends on the exact phrasing).
- The other two toy sentences score noticeably lower.

In [5]:
# [FULL]
# INSTRUCTOR DEMO: retrieval over 3 toy sentences -- embed, cosine, pick the closest.
toy_docs = [
    'The bank reported strong quarterly profits and grew its loan book.',          # 0: banking
    'The chipmaker shipped a new GPU and raised data-center revenue guidance.',     # 1: technology
    'The solar company commissioned new capacity and signed power agreements.',     # 2: energy
]
toy_query = 'How did the lender do this quarter?'

doc_vecs   = model.encode(toy_docs)
query_vec  = model.encode([toy_query])
sims       = cosine_similarity(query_vec, doc_vecs)[0]   # one score per doc

best_idx = int(np.argmax(sims))
print('Similarity of the query to each toy sentence:')
for i, (doc, s) in enumerate(zip(toy_docs, sims)):
    mark = '  <-- closest' if i == best_idx else ''
    print(f'  [{i}] {s:.3f}  "{doc}"{mark}')
print()
print('Retrieved sentence:', toy_docs[best_idx])
print('That is retrieval. The full RAG pipeline below does this over real earnings snippets, then generates an answer from the winner.')

Similarity of the query to each toy sentence:
  [0] 0.356  "The bank reported strong quarterly profits and grew its loan book."  <-- closest
  [1] 0.188  "The chipmaker shipped a new GPU and raised data-center revenue guidance."
  [2] 0.105  "The solar company commissioned new capacity and signed power agreements."

Retrieved sentence: The bank reported strong quarterly profits and grew its loan book.
That is retrieval. The full RAG pipeline below does this over real earnings snippets, then generates an answer from the winner.


## Step 3: Chunk the earnings text into passages

Each of the six snippets above is five sentences glued together. A whole snippet is a clumsy retrieval unit: if a question is about credit quality, we want to retrieve the *sentence* about non-performing loans, not the whole FirstBank paragraph that also talks about deposit spreads and capital ratios.

So we split each snippet into its individual sentences. Each sentence becomes one "chunk", tagged with the company it came from (that tag is what we will cite later). Real RAG systems chunk by paragraphs or fixed token windows; for our short transcripts, one sentence per chunk is the right grain.

In [6]:
# [FULL] Chunk each snippet into sentences. Each chunk carries its source company.
chunks = []        # list of sentence strings
chunk_source = []  # parallel list: which company each chunk came from

for company, snippet in zip(companies, snippets):
    # Split on '. ' and drop empties; re-attach the period so the text reads naturally.
    sentences = [s.strip() for s in snippet.split('. ') if s.strip()]
    for s in sentences:
        text = s if s.endswith('.') else s + '.'
        chunks.append(text)
        chunk_source.append(company)

print(f'{len(chunks)} chunks total ({len(companies)} companies x ~5 sentences each).')
print()
print('First few chunks:')
for i in range(4):
    print(f'  [{i}] ({chunk_source[i]}) {chunks[i]}')

30 chunks total (6 companies x ~5 sentences each).

First few chunks:
  [0] (TechCore) TechCore delivered record cloud revenue this quarter, growing 34 percent year-over-year.
  [1] (TechCore) Our AI inference platform saw adoption across 12 new enterprise clients.
  [2] (TechCore) Capital expenditure for data center expansion reached 2.1 billion dollars.
  [3] (TechCore) We guide for continued double-digit growth as enterprises accelerate digital transformation.


## Step 4: Embed every chunk

Same operation as Day 5: hand the list of chunk strings to `model.encode`, get back one 384-dimensional vector per chunk. These vectors are the "vector store": the chunks plus their embeddings, sitting in memory, waiting for a query. In a production system this would be a database (FAISS, Pinecone, pgvector); for about 30 chunks a NumPy array is plenty.

In [7]:
# [FULL] Embed all chunks once. Shape: (n_chunks, 384).
chunk_embeddings = model.encode(chunks)
print('Chunk embedding matrix shape:', chunk_embeddings.shape)
print('This is the vector store: every earnings sentence is now a point in 384-dimensional space.')
print('Chunking and embedding happen ONCE, offline. Everything below happens per question.')

Chunk embedding matrix shape: (30, 384)
This is the vector store: every earnings sentence is now a point in 384-dimensional space.
Chunking and embedding happen ONCE, offline. Everything below happens per question.


## CHECKPOINT 1: For the query "revenue growth", which snippets get retrieved, and what does the final prompt look like?

Before we run the retrieval cell, predict (discuss with your neighbour):

1. The query is `"what was revenue growth?"`. Several companies mention revenue growth (TechCore 34 percent, ChipMaker 58 percent data-center, SolarPower 41 percent). With `k=3`, which three chunks do you expect cosine similarity to pull, and from which companies?
2. Once those three chunks are retrieved, what does the prompt handed to the generator look like? Sketch it: how many chunks, where does the question go, what instruction wraps them?

Write your guess here, then run the next two cells:

```
Retrieved chunks (guess): ...
Prompt shape (guess): ...
```

## Step 5: The wrong way first -- retrieve ALL chunks into the prompt

Tempting shortcut: skip retrieval, dump *every* chunk into the prompt, let the model sort it out. Watch what that does to the prompt. We will build the all-chunks prompt and just measure its length and how on-topic it is.

In [8]:
# [MIN]
# WRONG WAY: stuff ALL chunks into the prompt instead of retrieving the relevant few.
question = 'What was revenue growth?'

all_chunks_context = '\n'.join(f'- ({src}) {c}' for src, c in zip(chunk_source, chunks))
bloated_prompt = (
    'Use the following earnings-call passages to answer the question.\n'
    f'Passages:\n{all_chunks_context}\n'
    f'Question: {question}\n'
    'Answer:'
)

# How many of the chunks actually mention revenue or growth? (a crude relevance check)
relevant = sum(('revenue' in c.lower() or 'grew' in c.lower() or 'growth' in c.lower()) for c in chunks)
print(f'All-chunks prompt: {len(chunks)} passages, about {len(bloated_prompt)} characters.')
print(f'Of those {len(chunks)} passages, only about {relevant} mention revenue or growth at all.')
print(f'So {len(chunks) - relevant} irrelevant passages are diluting the prompt, and a real model would also')
print('hit its context-window limit and silently truncate, often dropping the relevant part.')
print()
print('First 400 characters of the bloated prompt (note how much noise the model has to wade through):')
print(bloated_prompt[:400] + ' ...')

All-chunks prompt: 30 passages, about 2946 characters.
Of those 30 passages, only about 5 mention revenue or growth at all.
So 25 irrelevant passages are diluting the prompt, and a real model would also
hit its context-window limit and silently truncate, often dropping the relevant part.

First 400 characters of the bloated prompt (note how much noise the model has to wade through):
Use the following earnings-call passages to answer the question.
Passages:
- (TechCore) TechCore delivered record cloud revenue this quarter, growing 34 percent year-over-year.
- (TechCore) Our AI inference platform saw adoption across 12 new enterprise clients.
- (TechCore) Capital expenditure for data center expansion reached 2.1 billion dollars.
- (TechCore) We guide for continued double-digit  ...


### The fix: retrieve only the top-k most similar chunks

Retrieval exists precisely to make this selection. Embed the question, score every chunk by cosine similarity to it, keep the top `k` (we use `k=3`: a typical, small value). The prompt then holds three on-topic passages instead of thirty mixed ones. The model gets a clean task, the prompt stays short, and we know which chunks went in (so we can cite them).

<!-- # [[FULL]] -->

**Expected output (the top-k retrieval below):**

- A pandas DataFrame (or list) of 3 rows: the 3 chunks closest to the query.
- Each row carries: the chunk text, its source company tag (BankCo / TechCo / OilCo / RetailCo), and the cosine similarity.
- The top similarity is roughly 0.4 to 0.7 for a finance query; lower scores mean the query is off-topic for the corpus.

In [9]:
# [FULL] Retrieval: embed the question, cosine-similarity against all chunks, take the top k.
def retrieve(query, k=3):
    q_vec = model.encode([query])
    sims  = cosine_similarity(q_vec, chunk_embeddings)[0]   # one score per chunk
    top_idx = np.argsort(sims)[::-1][:k]                    # indices of the k highest scores
    return [(int(i), chunks[i], chunk_source[i], float(sims[i])) for i in top_idx]

question = 'What was revenue growth?'
hits = retrieve(question, k=3)

print(f'Query: "{question}"  (k=3)')
print('Retrieved chunks (most similar first):')
for rank, (idx, text, src, score) in enumerate(hits, start=1):
    print(f'  {rank}. [{src}] (score {score:.3f}) {text}')

Query: "What was revenue growth?"  (k=3)
Retrieved chunks (most similar first):
  1. [SolarPower] (score 0.567) Revenue grew 41 percent driven by long-term power purchase agreements.
  2. [ChipMaker] (score 0.513) Revenue from the data center segment grew 58 percent versus prior year.
  3. [TechCore] (score 0.436) Capital expenditure for data center expansion reached 2.1 billion dollars.


<!-- # [[FULL]] -->

## Mini-quiz: Embedding and retrieval

Discuss with your neighbour, then scroll to the **Answers (read after discussing)** cell below.

**Q1. Why do we embed chunks instead of comparing raw text?**

- **A.** Raw text comparison is too slow at scale.
- **B.** Embeddings let us measure semantic similarity, not just word overlap.
- **C.** Raw text takes more disk space than a 384-dimension vector.
- **D.** Both B and (in practice) C, but B is the core reason.

**Q2. Cosine similarity returns a value in what range for these embeddings?**

- **A.** From -1 to 1 in general; in practice, near 0 to near 1 for normalized embeddings of related text.
- **B.** From 0 to 100 percent.
- **C.** From 0 to infinity.
- **D.** Always exactly 0 or exactly 1.

**Q3. If we set k=3 instead of k=all, what is the main benefit?**

- **A.** The retrieval step runs faster.
- **B.** The prompt stays small and on-topic; the model is not distracted by irrelevant chunks.
- **C.** We save disk space on the embedding store.
- **D.** The cosine similarity becomes more accurate.


<!-- # [[FULL]] -->

## Answers (read after discussing)

**Q1: D.** Embeddings catch semantic similarity (the main pedagogical point), and vector storage is also more compact in practice.

**Q2: A.** Cosine similarity is defined on -1 to 1; for normalized sentence embeddings of related text it tends to sit between roughly 0 and 1.

**Q3: B.** Top-k keeps the prompt focused. Cosine similarity itself does not change with k; only what we feed the model does.


<!-- # [[FULL]] -->

### Section 2 terms (just-in-time recap)

- **Prompt**: the full text we hand to the language model = retrieved chunks + question.
- **Context window**: hard limit on prompt length; do not exceed.
- **Grounding**: forcing the answer to come from the prompt, not the model's training memory.

## Step 6: Build the grounded prompt -- only the retrieved chunks plus the question

Now we assemble the prompt the generator will see. It contains nothing but the three retrieved chunks (each tagged with its source company) and the question, wrapped in a short instruction telling the model to answer using only those passages. This is the "augmented" in Retrieval-Augmented Generation: the model's input is augmented with retrieved evidence.

In [10]:
# [FULL] Build the prompt: retrieved context + question. Nothing else goes in.
def build_prompt(question, hits):
    context = '\n'.join(f'- ({src}) {text}' for (_, text, src, _) in hits)
    return (
        'Answer the question using only the earnings-call passages below. '
        'Be concise. If the passages do not contain the answer, say so.\n'
        f'Passages:\n{context}\n'
        f'Question: {question}\n'
        'Answer:'
    )

prompt = build_prompt(question, hits)
print('THE PROMPT THE GENERATOR WILL SEE:')
print('-' * 60)
print(prompt)
print('-' * 60)
print(f'Prompt length: about {len(prompt)} characters -- compare with the about {len(bloated_prompt)}-character all-chunks prompt above.')
print(f'Source chunks that went in: {[src for (_, _, src, _) in hits]}')

THE PROMPT THE GENERATOR WILL SEE:
------------------------------------------------------------
Answer the question using only the earnings-call passages below. Be concise. If the passages do not contain the answer, say so.
Passages:
- (SolarPower) Revenue grew 41 percent driven by long-term power purchase agreements.
- (ChipMaker) Revenue from the data center segment grew 58 percent versus prior year.
- (TechCore) Capital expenditure for data center expansion reached 2.1 billion dollars.
Question: What was revenue growth?
Answer:
------------------------------------------------------------
Prompt length: about 440 characters -- compare with the about 2946-character all-chunks prompt above.
Source chunks that went in: ['SolarPower', 'ChipMaker', 'TechCore']


<!-- # [[FULL]] -->

## Mini-quiz: Prompt construction

Discuss with your neighbour, then scroll to the **Answers (read after discussing)** cell below.

**Q1. Why do we put the retrieved chunks BEFORE the question in the prompt?**

- **A.** Convention; either order works the same.
- **B.** Models tend to attend most strongly to the most recent text; the question goes last so it is fresh in the model's attention.
- **C.** It is the only order the model accepts.
- **D.** It saves tokens.

**Q2. What is the main failure mode of stuffing ALL chunks into the prompt?**

- **A.** The cosine similarities become wrong.
- **B.** The prompt may exceed the context window, and even if it fits, irrelevant chunks dilute the signal.
- **C.** The retrieval step crashes.
- **D.** Nothing fails; the model is robust to noise.

**Q3. Why does the prompt include the explicit instruction 'answer only from the passages above'?**

- **A.** It is required by the FastAPI framework.
- **B.** It is a grounding hint: it nudges the model away from using its training-time memory and toward the retrieved text.
- **C.** It tells the model which language to answer in.
- **D.** It prevents the model from outputting code.


<!-- # [[FULL]] -->

## Answers (read after discussing)

**Q1: B.** Recency bias in language models is real; placing the question last keeps it salient and reduces drift away from the user's intent.

**Q2: B.** Two compounding failures: overflowing the context window AND diluting relevance with off-topic chunks.

**Q3: B.** A grounding instruction is a soft constraint; it reduces hallucination but does not eliminate it.


<!-- # [[FULL]] -->

### Section 3 terms (just-in-time recap)

- **Generation**: the model writes a natural-language answer.
- **flan-t5-small**: the small text-to-text model we use here; runs locally on Colab CPU.
- **Citation**: the answer must point to the source chunk(s) so the user can verify.
- **Hallucination**: a generated fact the source does not contain; mitigated by RAG, not eliminated.

## Step 7: Generate the answer with a small local model

Now the generation half. We use `google/flan-t5-small` through `transformers.pipeline("text2text-generation")`: a roughly 300 MB model that runs on Colab's free CPU, no API key, no payment. It is small and not chatty, but it does what we need: read the grounded prompt, write a short answer from the passages in it.

If you ran the warm-start cell over the break, this model is already downloaded and the next cell is instant. If not, the first call pulls about 300 MB (a minute or two on a decent connection). Once it is loaded, it stays loaded for the rest of the session. The answer cell also prints which chunks the answer was built from, that is the citation.

In [12]:
from transformers import pipeline

generator = pipeline('text-generation', model='google/flan-t5-small')
print('Generator ready: google/flan-t5-small (text-generation, runs on CPU).')

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

Generator ready: google/flan-t5-small (text-generation, runs on CPU).


<!-- # [[FULL]] -->

**Expected output (the RAG pipeline below):**

- A short generated answer, 1 to 3 sentences, that mentions facts visible in the retrieved chunks.
- Followed by a `[source: <ChunkID>]` style tag (or printed list of chunk IDs) showing which chunks the answer came from.
- Style: concise, sometimes blunt - this is flan-t5-small, not GPT-4. The point is that the answer is grounded and traceable, not eloquent.

In [ ]:
# [FULL] The full RAG pipeline in one function: retrieve top-k -> build prompt -> generate -> cite.
def rag_answer(question, k=3, max_new_tokens=60):
    hits   = retrieve(question, k=k)
    prompt = build_prompt(question, hits)
    out    = generator(prompt, max_new_tokens=max_new_tokens)[0]['generated_text'].strip()
    sources = [f'{src} (chunk {idx})' for (idx, _, src, _) in hits]
    return out, sources, hits

answer, sources, hits = rag_answer('What was revenue growth?', k=3)
print('QUESTION: What was revenue growth?')
print('ANSWER  :', answer)
print('CITED FROM:', ', '.join(sources))
print()
print('Supporting passages (so the analyst can verify):')
for rank, (idx, text, src, score) in enumerate(hits, start=1):
    print(f'  {rank}. [{src}] {text}')
print()
print('Try another:')
answer2, sources2, _ = rag_answer('What did the banks say about credit quality?', k=3)
print('  Q: What did the banks say about credit quality?')
print('  A:', answer2)
print('  cited from:', ', '.join(sources2))

### Fallback: a degraded extractive responder (if the flan-t5-small download stalls)

If the `flan-t5-small` download stalled on a slow connection and the cells above did not run, this `# [MIN]` cell still gives you something to demo. It is a *degraded fallback*: instead of a generated sentence, it just returns the single top-retrieved snippet with a one-line framing. The retrieval half (the trusted half) is unchanged; only the generation half is replaced by "hand back the best passage verbatim". The real answer comes from the generator above; use this only if the model is unavailable.

In [ ]:
# [MIN] Degraded fallback: no language model, just return the top retrieved snippet.
def rag_answer_extractive(question, k=3):
    hits = retrieve(question, k=k)
    top_idx, top_text, top_src, top_score = hits[0]
    framed = f'Based on the {top_src} earnings call: "{top_text}"'
    sources = [f'{src} (chunk {idx})' for (idx, _, src, _) in hits]
    return framed, sources, hits

ans, srcs, hits = rag_answer_extractive('What was revenue growth?', k=3)
print('QUESTION : What was revenue growth?')
print('ANSWER   :', ans)
print('CITED FROM:', ', '.join(srcs))
print()
print('This is the degraded path: it returns the best passage verbatim instead of a generated answer.')
print('It still cites its source. Use it only if google/flan-t5-small could not be downloaded.')

## CHECKPOINT 2: If we set k=10 instead of k=3, what goes wrong?

Before you change anything, predict (discuss with your neighbour): we have only about 30 chunks total. If `retrieve(question, k=10)` pulls ten chunks into the prompt instead of three, what happens to (a) the prompt length, (b) how on-topic the prompt is, (c) the quality of the generated answer? When would a *larger* k actually help, and when does it hurt?

Your prediction: ________

*(Then run the comparison cell.)*

In [ ]:
# [FULL] Compare k=3 vs k=10: prompt length and how many retrieved chunks are actually relevant.
question = 'What was revenue growth?'
for k in (3, 10):
    hits_k   = retrieve(question, k=k)
    prompt_k = build_prompt(question, hits_k)
    rel = sum(('revenue' in t.lower() or 'grew' in t.lower() or 'growth' in t.lower()) for (_, t, _, _) in hits_k)
    print(f'k={k:2d}:  prompt about {len(prompt_k):4d} chars,  {rel}/{k} retrieved chunks mention revenue/growth,  sources={[s for (_, _, s, _) in hits_k]}')
print()
print('Bigger k = longer prompt + more off-topic chunks dragged in (the 4th to 10th best are weak matches here).')
print('A larger k helps only when the answer truly spans many passages; for a focused question it just adds noise')
print('and, in a real model with a finite context window, risks truncating the passages that mattered. Start at 3, tune by inspection.')

<!-- # [[FULL]] -->

## Mini-quiz: Generation

Discuss with your neighbour, then scroll to the **Answers (read after discussing)** cell below.

**Q1. Why use flan-t5-small instead of a larger model?**

- **A.** It writes better natural language than larger models.
- **B.** It is open-source, free, ~300 MB, and runs on Colab CPU; tradeoff is shorter and simpler answers.
- **C.** It is required by Hugging Face Spaces.
- **D.** Smaller models hallucinate less in general.

**Q2. What is the role of the # [MIN] extractive fallback cell?**

- **A.** It is the default path; the language model is the fallback.
- **B.** It is a degraded but guaranteed-working fallback if the flan-t5-small download stalls in class.
- **C.** It is a different model that is more accurate.
- **D.** It is dead code that should be removed.

**Q3. Why does the generated answer reference which chunk it came from?**

- **A.** For aesthetics.
- **B.** Citations let the user verify the answer against the source; an uncited RAG answer is back to the hallucination problem.
- **C.** It is required by Pydantic.
- **D.** It is a Gradio UI convention.


<!-- # [[FULL]] -->

## Answers (read after discussing)

**Q1: B.** Small + free + CPU-friendly is the point; expect concise answers, not polished prose. Smaller models can hallucinate MORE per token, not less.

**Q2: B.** The # [MIN] tag means 'non-droppable on minimum-viable path'; the fallback guarantees the lesson works even if the model download fails.

**Q3: B.** Trust is built by traceability. RAG without citations is just a confident guess engine.


<!-- # [[FULL]] -->

### Section 4 terms (just-in-time recap)

- **Hugging Face Space**: a free hosted environment that runs your `app.py` and gives a public URL.
- **requirements.txt**: the list of Python packages the Space installs before running the app.
- **Gradio**: a Python library that turns a function into a web UI (text box in, text box out).

## Step 8: Deploy the chatbot to a Hugging Face Space

A RAG pipeline in a notebook serves nobody. The next two cells write a tiny Gradio app (`app.py`) that wraps everything above, plus a `requirements.txt` listing what it needs. Then a real public URL is a five-minute browser job, no GitHub repo required.

<!-- # [[FULL]] -->

**Expected output (the `%%writefile app.py` cell below):**

- A confirmation line `Writing app.py` printed by Colab.
- A new `app.py` file in the Colab working directory containing the Gradio app definition.
- No model is downloaded or executed by `%%writefile` itself; it only writes the file.

In [13]:
%%writefile app.py
# [MIN] Gradio RAG earnings chatbot -- self-contained app for Hugging Face Spaces.
import numpy as np
import gradio as gr
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# --- The six Day-5 earnings snippets (same data as the notebook) ---
snippets = [
    ('TechCore delivered record cloud revenue this quarter, growing 34 percent year-over-year. '
     'Our AI inference platform saw adoption across 12 new enterprise clients. '
     'Capital expenditure for data center expansion reached 2.1 billion dollars. '
     'We guide for continued double-digit growth as enterprises accelerate digital transformation. '
     'Operating margins improved 200 basis points driven by software mix shift.'),
    ('ChipMaker reported strong demand for high-bandwidth memory in AI accelerator markets. '
     'Our next-generation GPU architecture enters mass production next quarter. '
     'Revenue from the data center segment grew 58 percent versus prior year. '
     'We are expanding fab capacity with a 4 billion dollar investment. '
     'Supply constraints in advanced packaging remain our primary near-term risk.'),
    ('FirstBank reported net interest income of 3.4 billion, up 8 percent on higher deposit spreads. '
     'Credit quality remains solid with non-performing loans at 0.7 percent of total portfolio. '
     'We grew our commercial lending book by 12 percent while maintaining disciplined underwriting. '
     'Tier 1 capital ratio stands at 13.2 percent, well above regulatory minimums. '
     'Provision for credit losses declined as charge-offs trended favorably.'),
    ('RegionalBank navigated a challenging rate environment with net interest margin compression. '
     'We added 42,000 new checking accounts, demonstrating strong retail franchise strength. '
     'Mortgage origination volume fell 23 percent as refinancing activity slowed. '
     'Cost efficiency ratio improved to 58 percent following branch optimization. '
     'We remain well-capitalized with CET1 ratio of 12.8 percent.'),
    ('OilGiant achieved production of 1.2 million barrels per day, exceeding guidance by 4 percent. '
     'Realized oil prices averaged 82 dollars per barrel, supporting strong free cash flow. '
     'We returned 2.8 billion dollars to shareholders through buybacks and dividends. '
     'Exploration success in the Gulf adds 200 million barrels to proven reserves. '
     'Refining margins benefited from tight distillate markets globally.'),
    ('SolarPower commissioned 3.2 gigawatts of new solar capacity across seven markets. '
     'Revenue grew 41 percent driven by long-term power purchase agreements. '
     'Our battery storage pipeline reached 18 gigawatt-hours, enabling round-the-clock delivery. '
     'Government incentives contributed 340 million dollars to project economics. '
     'We are on track to reach 20 gigawatts of total installed capacity by year-end.'),
]
companies = ['TechCore', 'ChipMaker', 'FirstBank', 'RegionalBank', 'OilGiant', 'SolarPower']

# --- Chunk into sentences (one chunk per sentence, tagged with its company) ---
chunks, chunk_source = [], []
for company, snippet in zip(companies, snippets):
    for s in [p.strip() for p in snippet.split('. ') if p.strip()]:
        chunks.append(s if s.endswith('.') else s + '.')
        chunk_source.append(company)

# --- Load models once at startup ---
embedder  = SentenceTransformer('paraphrase-MiniLM-L3-v2')

generator = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)
from transformers import pipeline

chunk_embeddings = embedder.encode(chunks)

def retrieve(query, k=3):
    q_vec = embedder.encode([query])
    sims  = cosine_similarity(q_vec, chunk_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    return [(int(i), chunks[i], chunk_source[i], float(sims[i])) for i in top_idx]

def build_prompt(question, hits):
    context = '\n'.join(f'- ({src}) {text}' for (_, text, src, _) in hits)
    return ('Answer the question using only the earnings-call passages below. '
            'Be concise. If the passages do not contain the answer, say so.\n'
            f'Passages:\n{context}\nQuestion: {question}\nAnswer:')

def chat(question):
    if not question or not question.strip():
        return 'Ask a question about the earnings calls.', ''
    hits   = retrieve(question, k=3)
    answer = generator(build_prompt(question, hits), max_new_tokens=60)[0]['generated_text'].strip()
    cited  = '\n'.join(f'[{src}] {text}' for (_, text, src, _) in hits)
    return answer, cited

demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label='Question about the earnings calls', placeholder='What was revenue growth?'),
    outputs=[gr.Textbox(label='Grounded answer'), gr.Textbox(label='Retrieved passages (the answer is based on these)')],
    title='Earnings Call RAG Chatbot',
    description='Retrieval-Augmented Generation over six synthetic earnings-call transcripts. The answer is generated only from the retrieved passages shown on the right.',
)

if __name__ == '__main__':
    demo.launch()

Writing app.py


In [14]:
%%writefile requirements.txt
sentence-transformers
transformers==4.45.2
torch
gradio

Writing requirements.txt


### Deploying to Hugging Face Spaces (browser only, no GitHub repo)

You now have two files on disk: `app.py` and `requirements.txt`. Here is the whole deploy, in the browser:

1. Go to `huggingface.co` and sign in (free account).
2. Click your profile, then **New Space**.
3. Give it a name (for example `earnings-rag-chatbot`), choose **Gradio** as the SDK, leave it public, and keep the free CPU hardware. Click **Create Space**.
4. On the new Space's page, open the **Files** tab, click **Add file**, then **Upload files**. Drag `app.py` and `requirements.txt` from your computer (download them from Colab's file panel first if needed) into the browser. Commit.
5. The Space starts building automatically: it reads `requirements.txt`, installs everything, downloads the two models (this first build takes a few minutes because of `flan-t5-small`), then runs `app.py`.
6. When the build finishes, the **App** tab shows your live chatbot at a public URL like `https://huggingface.co/spaces/your-name/earnings-rag-chatbot`. Share that link. Anyone can ask it questions.

No `Dockerfile`, no public GitHub repo, no payment. Just two files dragged into a browser.

*(If you happen to have your code in a public GitHub repo, Streamlit Community Cloud is an alternative: it deploys a Streamlit app on every push to a public repo. For a private course repo, Hugging Face Spaces drag-and-drop is the path with no friction.)*

<!-- # [[FULL]] -->

## Mini-quiz: Deploying to Hugging Face Spaces

Discuss with your neighbour, then scroll to the **Answers (read after discussing)** cell below.

**Q1. Why does the Hugging Face Spaces drag-and-drop path work without a GitHub repo?**

- **A.** HF Spaces clones from a Drive folder.
- **B.** HF Spaces hosts the files directly; you upload app.py and requirements.txt in the browser, and the Space builds the image itself.
- **C.** A GitHub repo is required; the comment is wrong.
- **D.** HF Spaces uses pre-built images for known notebooks.

**Q2. What goes in requirements.txt for this RAG chatbot?**

- **A.** Only flan-t5-small.
- **B.** sentence-transformers, transformers, torch, gradio - the runtime dependencies of app.py.
- **C.** Every package on the Colab machine.
- **D.** Nothing; HF Spaces auto-detects imports.

**Q3. Why pick Gradio (or Streamlit) over a custom HTML front end?**

- **A.** Custom HTML is faster.
- **B.** Gradio turns a Python function into a usable web UI in 3 lines; you ship the model logic, not the UI plumbing.
- **C.** Custom HTML supports more browsers.
- **D.** Gradio is the only library HF Spaces accepts.


<!-- # [[FULL]] -->

## Answers (read after discussing)

**Q1: B.** HF Spaces is a managed host: upload the two files, it builds an isolated container and serves it. No git required.

**Q2: B.** Only the imports app.py actually uses. Listing too much makes builds slow; listing too little makes them fail.

**Q3: B.** Gradio (and Streamlit) trade UI flexibility for shipping speed; ideal for demos and grounded chatbots.


## What you just built

- A vector store: the six Day-5 earnings snippets chunked into sentences and embedded with `paraphrase-MiniLM-L3-v2`.
- A retriever: embed a question, cosine-similarity it against every chunk, return the top `k=3`. (Not all chunks, which overflows and dilutes; the top few, which keeps the prompt small and on-topic.)
- A grounded prompt: only the retrieved chunks plus the question, wrapped in a short instruction.
- A generator: `google/flan-t5-small` via `transformers.pipeline`, writing the answer from that prompt, on free CPU, no API key. (Plus a degraded extractive fallback if the model download stalls.)
- A citation: every answer prints which chunks it came from, so the analyst can verify.
- A deployable app: `app.py` + `requirements.txt`, ready to drag into a Hugging Face Space for a live URL.

Retrieval is the half you can trust; generation is the half you grounded. Together they are an earnings chatbot a finance analyst could actually use.

**Homework (in the SOLVED notebook):** the CHECKPOINT answers are filled in there. Try deploying your own Space, and try swapping `k` and `max_new_tokens` to see how the answers change.